# Day3 제조 AI Agent 아키텍처 실습 해설 노트북

본 노트북은 교육용 가상 제조 시나리오(`DisplayEdu Fab`)를 기반으로 작성된 강의용 해설 자료입니다.
모든 설비 ID, 알람 코드, 라인 ID, 공정명은 가상 데이터이며 실제 사내 시스템과는 무관합니다.

## 0. 노트북 목적과 Day3 실습 방향

3일차 수업은 PostgreSQL 문법이나 FastMCP 라이브러리 사용법을 익히는 시간이 아닙니다.
3일차의 진짜 주제는 **Agent가 호출할 Tool을 어떻게 설계할 것인가** 입니다.

이 노트북은 `src/day3` 폴더에 있는 파이썬 파일들을 아키텍처 관점으로 읽기 위한 해설 자료이자, 수강생이 직접 코드 셀을 실행해 구조를 확인하는 실습 노트북입니다.

**노트북 운영 원칙을 정리합니다.**

- 노트북에서는 **fallback 중심으로 안전하게 구조를 확인**합니다.
- 실제 `src/day3` 실행과 강사 데모에서는 **FastMCP 모드를 기본 시나리오**로 설명합니다.
- DB 연결, FastMCP 서버, Streamlit 실행이 필요한 셀은 자동 실행하지 않고 "선택 실행" 또는 "강사 데모"로 표시합니다.
- 실행 성공보다 **Tool 분할, State 전달, 실행 기록 해석**을 이해하는 것이 더 중요합니다.

**수강생이 노트북을 마친 뒤 이해해야 할 핵심 질문은 다음과 같습니다.**

1. Day3의 7개 Tool은 어떤 기준으로 분할되어 있는가?
2. DB Tool과 RAG Tool은 무엇이 다른가?
3. CoordinatorAgent / DBInvestigationAgent / ManualRAGAgent / IncidentSummaryAgent는 각각 무엇을 책임지는가?
4. State에는 어떤 정보가 누적되며, Agent 간 무엇을 전달하는가?
5. FastMCP 모드와 fallback 모드는 각각 언제 사용하는가?

## 1. 실행 환경과 프로젝트 경로 확인

노트북은 **프로젝트 루트(`manufacturing_agent_project`)에서 실행**하는 것을 권장합니다.
Day3 모듈은 `src.day3.*` 경로로 import되므로 `sys.path`에 프로젝트 루트가 포함되어 있어야 합니다.

### 1-1. 프로젝트 루트와 `src/day3` 파일 목록 확인

이 셀은 현재 작업 폴더와 `src/day3` 폴더 존재 여부, 그리고 폴더 안의 `.py` 파일 목록을 확인합니다.
특정 사용자 PC 경로를 하드코딩하지 않으며, DB나 서버를 호출하지 않으므로 **자동 실행해도 안전**합니다.
강의에서는 "내가 지금 어느 폴더에서 노트북을 실행하고 있는가"를 가장 먼저 확인합니다.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
day3_dir = project_root / "src" / "day3"

print("현재 작업 폴더:", project_root)
print("src/day3 존재 여부:", day3_dir.exists())

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("sys.path 등록 여부:", str(project_root) in sys.path)

if day3_dir.exists():
    for path in sorted(day3_dir.glob("*.py")):
        print("-", path.name)
else:
    print("src/day3 폴더를 찾지 못했습니다. 노트북을 프로젝트 루트에서 실행하고 있는지 확인하세요.")

### 1-2. `src.day3` 모듈 import 가능 여부 확인

이 셀은 노트북 환경에서 `src.day3` 패키지가 정상 import되는지 확인합니다.
외부 자원에 접속하지 않으므로 **자동 실행해도 안전**합니다.
오류가 나면 메시지에 따라 "프로젝트 루트에서 실행 중인지"를 다시 확인합니다.

In [ ]:
try:
    from src.day3.manufacturing_mcp_client import SUPPORTED_TOOLS
    print("src.day3 import 확인 완료")
    print("지원 Tool 개수:", len(SUPPORTED_TOOLS))
except Exception as e:
    print("src.day3 import 중 오류가 발생했습니다.")
    print("현재 노트북을 프로젝트 루트에서 실행하고 있는지 확인하세요.")
    print("오류:", repr(e))

### 1-3. Day3 핵심 파일 존재 여부 점검

이 셀은 노트북에서 다룰 9개 핵심 파일이 모두 존재하는지 빠르게 점검합니다.
파일 시스템만 읽으므로 **자동 실행해도 안전**합니다.
강의에서는 "파일 단위로 본 Day3 폴더의 구성"을 한눈에 확인합니다.

In [ ]:
required_files = [
    "postgres_db_tool.py",
    "search_manual_tool.py",
    "manufacturing_mcp_client.py",
    "manufacturing_mcp_server.py",
    "tool_registry_fallback.py",
    "multi_agent_roles.py",
    "day3_mcp_tool_agent_v2.py",
    "mcp_tool_client_streamlit_app.py",
    "day3_multi_agent_streamlit_app.py",
]

for file_name in required_files:
    path = day3_dir / file_name
    print(f"{file_name}: {'있음' if path.exists() else '없음'}")

## 2. Day3 전체 아키텍처 한눈에 보기

Day3 실습은 "사용자 질문 한 줄"을 받아 다음 흐름을 따라 결과를 만듭니다.

```
사용자 질문
        │
        ▼
  CoordinatorAgent           ← equipment_id / alarm_code 추출
        │
        ▼
  DBInvestigationAgent       ← 교육용 제조 DB/Log Tool 호출 (5개)
        │
        ▼
  ManualRAGAgent             ← search_manual Tool 호출 (RAG → CSV fallback)
        │
        ▼
  IncidentSummaryAgent       ← DB 결과 + RAG 결과를 합쳐 요약 생성
        │
        ▼
  State / agent_steps / 결과 JSON·Markdown
        │
        ▼
  Day4: 평가 / Guardrail / Tool Plan 검증으로 확장
```

Tool 호출 경로는 두 가지입니다.

- **FastMCP 모드** : `manufacturing_mcp_client.py` → `manufacturing_mcp_server.py` (별도 프로세스에서 실행)
- **fallback 모드** : `tool_registry_fallback.py`가 Tool 이름을 Python 함수에 직접 연결

노트북은 fallback 중심으로 구조를 확인하고, FastMCP 모드는 강사 데모/선택 실습으로 설명합니다.

## 3. Day3 파일 역할 매핑표

| 파일명 | 아키텍처 계층 | 핵심 역할 | 강의에서 볼 관점 | 현업 적용 시 바꿀 부분 |
|---|---|---|---|---|
| `postgres_db_tool.py` | DB/Log Tool 계층 | 교육용 제조 DB의 6개 조회 함수 제공 | Tool 1개가 하나의 명확한 업무만 갖는지 | 권한 제어, row limit, 감사 로그, 접속 정보 보안 저장소 연동 |
| `search_manual_tool.py` | RAG Tool 계층 | 알람 조치 절차/기술 문서 검색 (Day2 RAG → CSV fallback) | DB Tool과 RAG Tool의 책임 분리 | 문서 접근 권한, 검색 인덱스 운영, 결과 필드 표준화 |
| `manufacturing_mcp_client.py` | MCP Client 계층 | Tool 목록·설명 정의, FastMCP/fallback 호출 라우팅 | `SUPPORTED_TOOLS`, `TOOL_DESCRIPTIONS`, `call_tool()` | 인증, 호출 로깅, 에러 정책, 재시도 |
| `manufacturing_mcp_server.py` | MCP Server 계층 | Python 함수를 `@mcp.tool`로 등록 | Tool 등록 구조, 입력 정규화(`normalize_blank_text`) | 권한 가드, 감사 로깅, 입력 검증 강화 |
| `tool_registry_fallback.py` | fallback Registry | MCP 없이 Tool 이름을 함수로 연결하는 단순 예제 | Tool 이름과 함수가 분리되어 있다는 원칙 | 운영 환경에서는 mock/staging Tool로 대체 |
| `multi_agent_roles.py` | Agent 계층 | Coordinator / DB / RAG / Summary 4개 Agent와 State 정의 | 역할 분담과 State 전달 | LangGraph 노드화, 조건부 라우팅, 휴먼 리뷰 |
| `day3_mcp_tool_agent_v2.py` | 실행 launcher | 사용자 질문 → Multi-Agent 실행 → 결과 JSON/Markdown 저장 | 통합 흐름, 결과 산출물 구조 | 운영 큐, 작업 ID, 추적 로그 연결 |
| `mcp_tool_client_streamlit_app.py` | 보조 실습 UI | Tool 단일 호출 / 7개 Tool 순차 호출 시각화 | Tool 호출의 입출력 구조 확인 | 권한 분리, 사용자 인증, 출력 데이터 마스킹 |
| `day3_multi_agent_streamlit_app.py` | 보조 실습 UI | Multi-Agent 실행 흐름과 State 시각화 | agent_steps, db_results, rag_results 확인 | 작업 이력 저장, 알림 연결 |

## 4. Tool 목록과 Tool Description 확인

Tool이란 **Agent가 호출할 수 있는 업무 기능 하나**입니다.
Day3 예제의 핵심 Tool은 다음 7개이며, 이름과 설명은 `manufacturing_mcp_client.py`의 `SUPPORTED_TOOLS`, `TOOL_DESCRIPTIONS`에 정의되어 있습니다.

### 4-1. `manufacturing_mcp_client.py`의 7개 Tool 목록 확인

이 셀은 `manufacturing_mcp_client.py`에 정의된 7개 Tool 이름과 설명을 확인합니다.
FastMCP 서버를 실행하지 않으므로 **자동 실행해도 안전**합니다.
강의에서는 Tool 이름과 Description이 LLM의 Tool 선택 기준이 된다는 점을 확인합니다.

In [ ]:
from src.day3.manufacturing_mcp_client import SUPPORTED_TOOLS, TOOL_DESCRIPTIONS

print("지원 Tool 개수:", len(SUPPORTED_TOOLS))
print()

for index, tool_name in enumerate(SUPPORTED_TOOLS, start=1):
    print(f"{index}. {tool_name}")
    print("   설명:", TOOL_DESCRIPTIONS.get(tool_name))

### 4-2. Tool을 DB Tool과 RAG Tool로 분류해 보기

이 셀은 7개 Tool을 **데이터 근거의 성격**(이력/수치 vs 문서)에 따라 분류합니다.
외부 자원에 접속하지 않으므로 **자동 실행해도 안전**합니다.
강의에서는 "같은 일을 두 Tool이 동시에 하지 않는가"를 점검합니다.

In [ ]:
db_tools = [
    "get_equipment_status",
    "get_recent_alarm_events",
    "get_process_status",
    "get_quality_metrics",
    "get_maintenance_history",
    "get_equipment_overview",
]

rag_tools = [
    "search_manual",
]

print("[DB/Log Tool]")
for name in db_tools:
    print("-", name)

print("\n[RAG Tool]")
for name in rag_tools:
    print("-", name)

**설계 관점 질문**

- 이 Tool은 **하나의 명확한 업무 목적**만 갖는가?
- LLM이 "언제 이 Tool을 선택해야 하는지" 설명이 충분한가?
- DB 조회 Tool과 문서 검색 Tool이 **이름과 설명만 봐도 구분되는가**?
- 통합 Tool(`get_equipment_overview`)은 편리하지만, 다른 6개 Tool의 자리를 빼앗을 위험은 없는가?

## 5. DB/Log Tool 구조: `postgres_db_tool.py`

이 파일은 **교육용 제조 DB/Log 조회 기능을 Python 함수로 분리**한 예제입니다.
SQL 학습이 아니라 "DB 기능을 Agent Tool로 감싸는 구조"를 보는 것이 핵심입니다.

개별 DB Tool 6개와 통합 Overview Tool을 의도적으로 구분해 두었습니다.

### 5-1. `postgres_db_tool.py` 함수 signature 확인

이 셀은 DB Tool 함수 6종의 입력 구조(파라미터 이름과 타입)를 확인합니다.
**실제 DB 연결 함수는 호출하지 않습니다** (`inspect.signature`만 사용).
DB나 서버 없이도 동작하므로 **자동 실행해도 안전**합니다.
강의에서는 "각 Tool이 받는 입력이 충분히 좁게 정의되어 있는가"를 확인합니다.

In [ ]:
import inspect
import src.day3.postgres_db_tool as db_tool

db_tool_names = [
    "get_equipment_status",
    "get_recent_alarm_events",
    "get_process_status",
    "get_quality_metrics",
    "get_maintenance_history",
    "get_equipment_overview",
]

for name in db_tool_names:
    func = getattr(db_tool, name)
    print(f"{name}{inspect.signature(func)}")

### 5-2. `get_equipment_overview`가 "큰 Tool"인 이유 확인

이 셀은 `get_equipment_overview` 함수의 **앞부분(최대 1500자)** 만 소스로 가져와 확인합니다.
DB에 접속하지 않으므로 **자동 실행해도 안전**합니다.
전체 소스를 길게 출력하지 않도록 출력 길이를 제한했습니다.

In [ ]:
import inspect
import textwrap
from src.day3.postgres_db_tool import get_equipment_overview

source = inspect.getsource(get_equipment_overview)
print(textwrap.dedent(source[:1500]))

`get_equipment_overview`는 설비·알람·공정·품질·정비 정보를 한 번에 묶어 조회하는 통합 Tool입니다.
편리하지만 **하나의 Tool이 떠안는 책임이 커질 수 있으므로**, 현업 적용 시에는 통합 Tool로 둘지 개별 Tool로 나눌지 명확하게 판단해야 합니다.

**설계 관점 질문**

- 입력값은 충분히 제한되어 있는가? (예: `equipment_id`, `limit` 위주)
- 반환 데이터의 크기와 범위가 과도하지 않은가?
- 현업에서는 **row limit, 권한, 감사 로그**가 반드시 필요한가?
- DB 비밀번호, 토큰은 보안 저장소(Vault, Secret Manager)로 옮길 수 있는가?

## 6. RAG Tool 구조: `search_manual_tool.py`

`search_manual`은 **기술 문서/조치 절차를 검색하는 RAG Tool**입니다.
내부 동작은 다음 순서를 따릅니다.

1. `alarm_code`, `symptom`, `user_query`를 하나의 검색 쿼리로 합칩니다.
2. Day2 RAG (`src.day2.rag_search.search_top_k`)를 우선 시도합니다.
3. RAG가 실패하거나 결과가 없으면 `outputs/day2/chunk_metadata.csv` 등에서 **키워드 매칭 fallback**을 수행합니다.
4. 결과를 `{"query", "search_mode", "top_k", "result_count", "results"}` 형태의 dict로 반환합니다.

DB Tool은 **수치/이력 데이터**, RAG Tool은 **문서 근거**를 다룬다는 점을 기억합니다.

### 6-1. `search_manual_tool.py`의 검색 질의 생성 확인

이 셀은 RAG Tool이 여러 입력값(`alarm_code`, `symptom`, `user_query`)을 **하나의 검색 질의로 합치는 방식**을 확인합니다.
RAG나 CSV를 실제로 검색하지 않고 문자열 결합만 수행하므로 **자동 실행해도 안전**합니다.
강의에서는 "입력 슬롯을 어떻게 검색 질의로 정규화할 것인가"를 확인합니다.

In [ ]:
from src.day3.search_manual_tool import build_search_query

query = build_search_query(
    alarm_code="ALM-TEMP-402",
    symptom="temperature abnormal",
    user_query="조치 절차를 확인해 주세요.",
)

print("생성된 검색 질의:")
print(query)

### 6-2. `search_manual` 함수 signature 확인

이 셀은 `search_manual` Tool의 입력 시그니처를 확인합니다.
함수 객체의 메타 정보만 읽으므로 **자동 실행해도 안전**합니다.
강의에서는 "기본값(`top_k=3`)이 너무 크거나 작지 않은가"를 점검합니다.

In [ ]:
import inspect
from src.day3.search_manual_tool import search_manual

print(inspect.signature(search_manual))

### 6-3. `search_manual` 실제 호출 (선택 실행)

아래 셀은 **선택 실행**입니다. Day2 RAG 또는 `outputs/day2/chunk_metadata.csv`가 준비되어 있을 때만 의미가 있습니다.
결과가 비어 있어도 오류가 아니라, **준비된 데이터 상태에 따라 달라지는 정상 동작**입니다.
결과 dict의 `search_mode`가 `rag`인지 `fallback_csv`인지로 어떤 경로가 사용되었는지 확인할 수 있습니다.

In [ ]:
# 선택 실행:
# Day2 RAG 또는 CSV metadata가 준비되어 있을 때만 실행합니다.
# 결과가 비어 있어도 오류가 아니라 준비된 데이터 상태에 따라 달라질 수 있습니다.

# from src.day3.search_manual_tool import search_manual
#
# result = search_manual(
#     alarm_code="ALM-TEMP-402",
#     symptom="temperature abnormal",
#     top_k=3,
# )
#
# result

## 7. FastMCP Server/Client 구조

- `manufacturing_mcp_server.py` : Python 함수를 `@mcp.tool`로 등록하는 역할
- `manufacturing_mcp_client.py` : Tool 이름과 입력값으로 Tool을 호출하는 역할

핵심은 FastMCP 라이브러리 문법이 아니라 **Tool 등록 → Tool 호출 → 결과 구조 확인**의 흐름입니다.
Client는 `MCP_MODE` 환경변수에 따라 동작합니다.

- `MCP_MODE=fastmcp` : FastMCP 서버를 통해 Tool 호출
- `MCP_MODE=fallback` : `tool_registry_fallback.call_tool`로 호출
- `MCP_MODE=auto` (또는 미설정) : FastMCP를 먼저 시도하고 실패하면 fallback으로 전환

### 7-1. `manufacturing_mcp_server.py`의 등록 Tool 이름 확인

이 셀은 FastMCP 서버 모듈이 가지고 있는 `REGISTERED_TOOL_NAMES` 상수를 통해 **서버에 등록될 Tool 이름 목록**만 확인합니다.
**서버를 실행하지 않고 모듈만 import**하므로 자동 실행해도 안전합니다.
강의에서는 "Client가 부르는 Tool 목록(`SUPPORTED_TOOLS`)과 Server가 등록하는 Tool 목록이 일치하는가"를 확인합니다.

> 주의: `fastmcp` 라이브러리가 설치되어 있어야 import가 성공합니다. ImportError가 나면 가상환경 설정을 점검합니다.

In [ ]:
from src.day3.manufacturing_mcp_server import REGISTERED_TOOL_NAMES

print("FastMCP Server 등록 Tool 개수:", len(REGISTERED_TOOL_NAMES))

for name in REGISTERED_TOOL_NAMES:
    print("-", name)

### 7-2. Client가 노출하는 Tool 카탈로그 확인

이 셀은 `list_tools()`로 Client가 노출하는 Tool 카탈로그(이름 + 설명)를 dict 형태로 확인합니다.
외부 호출을 하지 않으므로 **자동 실행해도 안전**합니다.

In [ ]:
from src.day3.manufacturing_mcp_client import list_tools

tools_info = list_tools()
tools_info

### FastMCP 실행 명령 (선택 실행 / 강사 데모)

노트북 안에서 실행하지 않습니다. 별도 터미널 두 개를 사용합니다.

**터미널 1 — FastMCP 서버 실행**

```powershell
cd C:\work\manufacturing_agent_project
conda activate manufacturing_agent_env
python src/day3/manufacturing_mcp_server.py
```

**터미널 2 — Client 실행 (서버가 정상 기동된 뒤)**

```powershell
cd C:\work\manufacturing_agent_project
conda activate manufacturing_agent_env
$env:MCP_MODE="fastmcp"
$env:MCP_FASTMCP_URL="http://127.0.0.1:8765/mcp"
python src/day3/manufacturing_mcp_client.py
```

## 8. fallback Registry 구조: `tool_registry_fallback.py`

`tool_registry_fallback.py`는 **MCP 실행이 어려울 때 Tool 이름과 Python 함수를 직접 연결하는 단순 예비 구조**입니다.

중요한 점은 다음과 같습니다.

- 이 예제는 **모든 7개 Tool을 완전히 대체하는 운영용 구조가 아닙니다.**
- 현재 구조는 `get_equipment_overview`와 `search_manual` **두 개를 중심으로 한 예비 호출 구조**입니다.
- Multi-Agent 흐름에서 fallback 모드를 선택하면 DB 세부 조회는 `get_equipment_overview` 한 번으로 묶이는 형태가 됩니다.

### 8-1. fallback registry에 등록된 Tool 확인

이 셀은 `build_tool_registry()`로 fallback registry를 만든 뒤, **실제 등록된 Tool 이름만** 확인합니다.
실제 `call_tool()` 호출은 하지 않으므로 **자동 실행해도 안전**합니다.
강의에서는 "왜 7개가 아니라 2개인가"를 함께 이야기합니다.

In [ ]:
from src.day3.tool_registry_fallback import build_tool_registry

registry = build_tool_registry()

print("fallback registry 등록 Tool:")
for name in registry.keys():
    print("-", name)

현재 fallback registry는 모든 7개 Tool을 완전히 대체하지 않습니다.
`get_equipment_overview`, `search_manual` 중심의 **단순 예비 호출 구조**로 이해합니다.

### 8-2. fallback `call_tool` 실제 호출 (선택 실행)

아래 셀은 **선택 실행**입니다.
- `search_manual`은 RAG/CSV 준비 상태에 따라 결과가 달라질 수 있습니다.
- `get_equipment_overview`는 PostgreSQL 연결이 필요합니다.

노트북에서는 자동 실행하지 않으며, 강사 데모 또는 환경이 준비된 상태에서만 주석을 풀어 실행합니다.

In [ ]:
# 선택 실행:
# search_manual은 RAG/CSV 준비 상태에 따라 결과가 달라질 수 있습니다.
# get_equipment_overview는 DB 연결 환경이 필요할 수 있습니다.

# from src.day3.tool_registry_fallback import call_tool
#
# result = call_tool(
#     "search_manual",
#     {
#         "alarm_code": "ALM-TEMP-402",
#         "symptom": "temperature abnormal",
#         "top_k": 3,
#     },
# )
#
# result

**설계 관점 질문**

- fallback은 **운영용 대체 구조**인가, **교육용 예비 구조**인가?
- fallback에서도 Tool 이름과 입력 구조를 유지해야 하는 이유는 무엇인가?
- 현업에서는 fallback 자리를 어떤 환경으로 채울 것인가? (mock, sandbox, staging 등)

## 9. Multi-Agent 역할 분담: `multi_agent_roles.py`

이 파일은 **Agent 역할 분담과 State 전달 구조**를 보여줍니다.
LangGraph를 직접 사용하지 않지만, 각 클래스를 LangGraph의 Node로 해석할 수 있습니다.

| 클래스 | 역할 |
|---|---|
| `CoordinatorAgent` | 사용자 요청에서 `equipment_id`, `alarm_code`를 정규식으로 추출 |
| `DBInvestigationAgent` | 교육용 DB Tool 5종(`get_equipment_status` 등)을 순차 호출 |
| `ManualRAGAgent` | `search_manual` Tool 호출, 결과를 `rag_results`에 저장 |
| `IncidentSummaryAgent` | DB 결과와 RAG 결과를 합쳐 교육용 장애 대응 요약 생성 |

### 9-1. 초기 State 구조 확인

이 셀은 `create_initial_state()`로 만든 **Agent 간 공유 State**의 초기 구조를 확인합니다.
DB나 FastMCP를 호출하지 않으므로 **자동 실행해도 안전**합니다.
강의에서는 "공유해야 하는 정보만 State에 들어가 있는가"를 점검합니다.

In [ ]:
from src.day3.multi_agent_roles import create_initial_state

sample_query = "EQP-EV-03에서 ALM-TEMP-402 알람이 반복 발생했습니다. 원인과 조치 방향을 알려주세요."

state = create_initial_state(sample_query)
state

### 9-2. State key 설명을 코드로 확인

이 셀은 State의 **각 key와 값 타입**을 한 줄씩 출력합니다.
외부 호출이 없으므로 **자동 실행해도 안전**합니다.
강의에서는 "어떤 key가 초기에 비어 있고, 어느 Agent가 채우는가"를 매핑합니다.

In [ ]:
for key, value in state.items():
    print(f"{key}: {type(value).__name__} = {value}")

### 9-3. `equipment_id`, `alarm_code` 추출 함수 확인

이 셀은 사용자 요청 문자열에서 설비 ID와 알람 코드를 추출하는 **정규식 함수**의 동작을 확인합니다.
외부 자원 없이 문자열만 처리하므로 **자동 실행해도 안전**합니다.
강의에서는 "패턴이 바뀐 입력에서는 정규식 추출이 깨질 수 있다"는 점을 확인합니다.

In [ ]:
from src.day3.multi_agent_roles import extract_equipment_id, extract_alarm_code

sample_text = "EQP-EV-03에서 ALM-TEMP-402 알람이 반복 발생했습니다."

print("equipment_id:", extract_equipment_id(sample_text))
print("alarm_code:", extract_alarm_code(sample_text))

### 9-4. `CoordinatorAgent`만 안전하게 실행해 보기

이 셀은 4개 Agent 중 **DB Tool/RAG Tool을 호출하지 않는** `CoordinatorAgent`만 실행합니다.
DB나 FastMCP가 없는 환경에서도 동작하므로 **자동 실행해도 안전**합니다.
강의에서는 "Agent 하나가 State에 어떻게 변경 사항을 누적하는가"를 직접 눈으로 확인합니다.

In [ ]:
from src.day3.multi_agent_roles import CoordinatorAgent, create_initial_state

state = create_initial_state("EQP-EV-03에서 ALM-TEMP-402 알람이 반복 발생했습니다.")
state = CoordinatorAgent().run(state)

state

### 9-5. Agent 클래스 목록 확인

이 셀은 `multi_agent_roles.py`에 정의된 클래스 중 **이름이 `Agent`로 끝나는 클래스**를 자동으로 모아 보여줍니다.
`inspect.getmembers`만 사용하므로 **자동 실행해도 안전**합니다.
강의에서는 "Agent 클래스 4개가 모두 등장하는가"를 확인합니다.

In [ ]:
import inspect
import src.day3.multi_agent_roles as roles

agent_classes = []

for name, obj in inspect.getmembers(roles, inspect.isclass):
    if name.endswith("Agent"):
        agent_classes.append(name)

agent_classes

**설계 관점 질문**

- Agent 간 전달해야 하는 **최소 정보**는 무엇인가? (`equipment_id`, `alarm_code`, `line_id`)
- `db_results`와 `rag_results`를 분리하는 이유는 무엇인가? (출처 추적, 평가 분리, 4일차 Guardrail 연결)
- `agent_steps`는 어떤 실행 기록 역할을 하는가? (사람이 읽는 trace, 4일차 평가 입력)

## 10. Multi-Agent 흐름을 LangGraph 관점으로 해석하기

LangGraph 코드를 새로 작성하지 않아도, 현재 Multi-Agent 흐름은 LangGraph 노드 구조로 해석할 수 있습니다.

### 10-1. Multi-Agent 흐름을 LangGraph Node로 매핑하는 dict

이 셀은 실제 LangGraph 코드를 실행하지 않고, **현재 Multi-Agent 흐름을 LangGraph 관점으로 해석한 매핑표**를 dict로 만듭니다.
외부 자원 없이 dict만 만들므로 **자동 실행해도 안전**합니다.
강의에서는 이 매핑을 보며 "어떤 노드에 어떤 조건부 엣지가 필요한가"를 설계합니다.

In [ ]:
langgraph_mapping = {
    "CoordinatorAgent": "요청 분석 Node",
    "DBInvestigationAgent": "DB/Log Tool 호출 Node",
    "ManualRAGAgent": "RAG Tool 호출 Node",
    "IncidentSummaryAgent": "결과 종합 Node",
    "State": "Node 간 공유 작업 정보",
    "agent_steps": "실행 흐름 기록",
}

langgraph_mapping

**설계 관점 질문**

- 어떤 조건에서 DB Tool Node로 보내야 하는가? (`equipment_id`가 추출되었을 때)
- 어떤 조건에서 RAG Tool Node로 보내야 하는가? (`alarm_code` 또는 증상 키워드가 있을 때)
- 정보가 부족할 때는 어디로 보내야 하는가? (재질문 Node, Human Review Node)
- 위험 요청이 들어오면 Summary 전에 Guardrail Node가 차단해야 하는가?

## 11. 통합 실행 구조: `day3_mcp_tool_agent_v2.py`

이 파일은 **Day3 통합 실행 launcher**입니다. 실제 Multi-Agent 역할 분담은 `multi_agent_roles.py`가 담당합니다.

launcher가 하는 일은 다음과 같습니다.

1. 기본 사용자 질문(`DEFAULT_USER_QUERY`) 준비
2. `run_multi_agent_flow(user_query, mode=DEFAULT_MODE)` 호출
3. 결과 State를 `outputs/day3/day3_mcp_tool_agent_v2_result.json`으로 저장
4. `save_result_markdown(state)`로 Markdown 결과 저장 호출
5. 생성된 파일 경로를 콘솔에 출력

### 11-1. `day3_mcp_tool_agent_v2.py`의 주요 상수 확인

이 셀은 launcher 모듈의 **기본 실행 모드와 기본 사용자 질문 상수**를 확인합니다.
Multi-Agent 흐름을 실행하지 않고 모듈 import만 하므로 **자동 실행해도 안전**합니다.
강의에서는 "기본 모드가 무엇인가"와 "환경변수보다 코드 상수가 우선되는지"를 함께 확인합니다.

In [ ]:
import src.day3.day3_mcp_tool_agent_v2 as launcher

print("DEFAULT_MODE:", launcher.DEFAULT_MODE)
print("DEFAULT_USER_QUERY:", launcher.DEFAULT_USER_QUERY)

`day3_mcp_tool_agent_v2.py`는 통합 실행 launcher입니다.
**기본 실행 모드가 `fastmcp`** 이므로, 실제 실행 전 FastMCP 서버가 준비되어 있는지 확인해야 합니다.

### 11-2. 출력 폴더 예상 경로 확인

이 셀은 통합 launcher가 결과를 저장할 **출력 폴더의 예상 경로**와 존재 여부만 확인합니다.
**폴더를 생성하지 않습니다** (launcher의 `get_output_dir()`는 폴더를 만들 수 있으므로 직접 호출하지 않습니다).
외부 자원에 접속하지 않으므로 **자동 실행해도 안전**합니다.

In [ ]:
from pathlib import Path

output_dir = Path.cwd() / "outputs" / "day3"
print("Day3 출력 폴더 예상 경로:", output_dir)
print("현재 존재 여부:", output_dir.exists())

### 11-3. 통합 실행 (선택 실행 / 강사 데모)

아래 셀은 **선택 실행**입니다.
FastMCP 서버가 먼저 실행되어 있어야 정상 동작하며, 강의 중에는 Streamlit Tool Client 또는 fallback 구조 확인 후에 실행합니다.
실행하면 `outputs/day3/` 아래에 JSON과 Markdown 결과 파일이 생성됩니다.

In [ ]:
# 선택 실행:
# FastMCP 서버가 먼저 실행되어 있어야 정상 동작할 수 있습니다.
# 강의 중에는 Streamlit Tool Client 또는 fallback 구조 확인 후 실행합니다.

# from src.day3.day3_mcp_tool_agent_v2 import run_day3_agent_v2
#
# state = run_day3_agent_v2(
#     user_query="EQP-EV-03에서 ALM-TEMP-402 알람이 반복 발생했습니다. 원인과 조치 방향을 알려주세요.",
#     mode="fastmcp",
# )
#
# state

### 통합 실행 명령 (강사 데모용)

**FastMCP 모드** — 서버가 별도 터미널에서 이미 기동되어 있어야 합니다.

```powershell
cd C:\work\manufacturing_agent_project
conda activate manufacturing_agent_env
python src/day3/day3_mcp_tool_agent_v2.py
```

**fallback 환경변수를 사용하는 경우** (선택 실행 / 확인 필요):

```powershell
# 선택 실행/확인 필요:
# launcher가 MCP_MODE 환경변수를 실제로 반영하는지 확인한 뒤 실행합니다.
cd C:\work\manufacturing_agent_project
conda activate manufacturing_agent_env
$env:MCP_MODE="fallback"
python src/day3/day3_mcp_tool_agent_v2.py
```

## 12. Streamlit 보조 실습

Streamlit은 핵심 학습 대상이 아닙니다. **Tool 호출 결과와 Multi-Agent 흐름을 화면으로 확인하는 보조 도구**일 뿐입니다.
노트북에서는 Streamlit을 자동 실행하지 않으며, 실행 명령은 Markdown으로만 안내합니다.

| 파일 | 화면에서 확인할 항목 |
|---|---|
| `mcp_tool_client_streamlit_app.py` | Tool 단일 호출 / 7개 Tool 순차 호출, Tool 이름·입력값·결과 dict |
| `day3_multi_agent_streamlit_app.py` | 추출 정보, Agent 실행 단계, 최종 요약, DB/RAG 결과 |

### fallback 모드 실행 (안정 시나리오)

```powershell
cd C:\work\manufacturing_agent_project
conda activate manufacturing_agent_env
$env:MCP_MODE="fallback"
streamlit run src/day3/mcp_tool_client_streamlit_app.py
```

```powershell
cd C:\work\manufacturing_agent_project
conda activate manufacturing_agent_env
$env:MCP_MODE="fallback"
streamlit run src/day3/day3_multi_agent_streamlit_app.py
```

### FastMCP 모드 실행 (선택 실행 / 강사 데모)

FastMCP 서버가 별도 터미널에서 기동되어 있어야 합니다.

```powershell
cd C:\work\manufacturing_agent_project
conda activate manufacturing_agent_env
$env:MCP_MODE="fastmcp"
$env:MCP_FASTMCP_URL="http://127.0.0.1:8765/mcp"
streamlit run src/day3/mcp_tool_client_streamlit_app.py
```

## 13. 실행 기록을 Trace 관점으로 해석하기

Day3 예제에서는 별도의 `mcp_call_trace.jsonl` 파일이 반드시 생성된다고 가정하지 않습니다.
대신 다음 항목을 **"이번 실행에서 일어난 일"** 을 보여 주는 실행 기록으로 해석합니다.

| 실행 기록 항목 | 어디서 확인 | 해석 방법 |
|---|---|---|
| `agent_steps` | State, Markdown 결과 | 어떤 Agent가 어떤 단계를 수행했는지 순서대로 확인 |
| `db_results` | State, JSON 결과 | DB/Log Tool 호출 결과. Tool 이름별로 분리 |
| `rag_results` | State, JSON 결과 | RAG Tool 호출 결과. `search_mode`로 경로 확인 |
| `final_summary` | State, Markdown 결과 | DB·RAG 결과를 합친 교육용 장애 대응 요약 |
| `outputs/day3/*.json`, `*.md` | 파일 시스템 | 한 번의 실행이 남기는 산출물 묶음 |

### 13-1. 실행 기록 해석용 State 샘플 만들기

이 셀은 **실제 Tool 호출 없이** 설명용 `sample_state_for_review` dict를 만듭니다.
DB나 FastMCP가 없어도 동작하므로 **자동 실행해도 안전**합니다.
강의에서는 "진짜 State가 이 모양으로 누적되었다고 가정하면, 어떻게 해석할 것인가"를 연습합니다.

In [ ]:
sample_state_for_review = {
    "user_query": "EQP-EV-03에서 ALM-TEMP-402 알람이 반복 발생했습니다.",
    "equipment_id": "EQP-EV-03",
    "alarm_code": "ALM-TEMP-402",
    "line_id": "LINE-07",
    "db_results": {
        "equipment_status": "설비 기본 정보 조회 결과",
        "recent_alarm_events": "최근 알람 이력 조회 결과",
    },
    "rag_results": {
        "search_mode": "rag 또는 fallback_csv",
        "result_count": 3,
    },
    "agent_steps": [
        "CoordinatorAgent: 설비 ID와 알람 코드를 확인했습니다.",
        "DBInvestigationAgent: DB Tool을 호출했습니다.",
        "ManualRAGAgent: 기술 문서 검색 Tool을 호출했습니다.",
        "IncidentSummaryAgent: 결과 요약을 생성했습니다.",
    ],
    "final_summary": "교육용 장애 대응 요약",
}

sample_state_for_review

### 13-2. 실행 기록 key 확인

이 셀은 `sample_state_for_review`에서 **실행 기록으로 해석할 수 있는 key들**만 추려서 출력합니다.
외부 자원 없이 dict만 읽으므로 **자동 실행해도 안전**합니다.
강의에서는 "4일차 평가/Guardrail이 어떤 key를 입력으로 사용할 수 있는가"로 자연스럽게 연결합니다.

In [ ]:
execution_record_keys = [
    "agent_steps",
    "db_results",
    "rag_results",
    "final_summary",
]

for key in execution_record_keys:
    print(f"{key}: {sample_state_for_review.get(key)}")

**설계 관점 질문**

- 어떤 Tool이 어떤 순서로 호출되었는가?
- 입력값은 과도하지 않았는가? (예: 사용자 PII, 너무 큰 limit)
- DB 근거와 문서 근거가 결과에서 **분리 가능한 형태**로 저장되어 있는가?
- State 전달 과정에서 정보 누락은 없는가? (특히 `line_id`)
- 4일차에서 평가/Guardrail을 붙인다면 어떤 항목을 입력으로 사용할 것인가?

## 14. 현업 적용 체크리스트

교육용 예제를 현업으로 옮길 때 반드시 점검해야 하는 항목입니다.

| # | 점검 항목 | 확인 포인트 |
|---|---|---|
| 1 | DB 접속 정보 보안 | 비밀번호/토큰을 보안 저장소(Vault, Secret Manager)에 두는가? |
| 2 | Tool 권한 분리 | Tool별 호출 권한이 역할(role)별로 정의되어 있는가? |
| 3 | read-only vs action Tool | 조회 Tool과 실제 동작을 일으키는 Tool이 명확히 분리되어 있는가? |
| 4 | 반환 데이터 최소화 | 응답에 불필요한 컬럼/행이 포함되지 않는가? |
| 5 | RAG 문서 접근 권한 | 사용자 권한에 따라 검색 가능한 문서를 제한하는가? |
| 6 | State 최소 전달 | State에 PII나 불필요한 원본 데이터를 담지 않는가? |
| 7 | 실행 기록 감사성 | `agent_steps`/Tool 호출 로그가 감사 및 평가에 사용 가능한가? |
| 8 | fallback과 운영 구조 분리 | 교육용 fallback이 운영 코드 경로에 섞여 들어가지 않는가? |
| 9 | 외부 도구 입력 안전 | Claude Code 등 외부 도구에 실제 사내 데이터·API Key를 절대 입력하지 않는가? |

## 15. 강의 마무리 — 오늘 확인한 핵심 구조 요약

노트북을 닫기 전 오늘 확인한 내용을 한 표로 정리합니다.

| 구분 | 내용 |
|---|---|
| 7개 Tool 목록 | `get_equipment_status`, `get_recent_alarm_events`, `get_process_status`, `get_quality_metrics`, `get_maintenance_history`, `get_equipment_overview`, `search_manual` |
| DB Tool / RAG Tool 구분 | DB Tool 6개(이력·수치 데이터) vs RAG Tool 1개(문서 근거 검색) |
| Multi-Agent 역할 | CoordinatorAgent(요청 분석) / DBInvestigationAgent(DB 호출) / ManualRAGAgent(문서 검색) / IncidentSummaryAgent(요약) |
| State 실행 기록 항목 | `agent_steps`, `db_results`, `rag_results`, `final_summary`, `outputs/day3/*.json`, `*.md` |
| FastMCP vs fallback | FastMCP = Tool 등록/호출 표준 구조 / fallback = `get_equipment_overview`·`search_manual` 중심의 단순 예비 호출 구조 |
| 현업 적용 시 주요 수정 포인트 | 비밀 정보 보안 저장소화, Tool 권한 분리, read-only vs action 구분, 반환 데이터 최소화, RAG 문서 접근 권한, 감사 로그 |

수강생은 노트북을 닫기 전에 다음 한 문장을 스스로 채워 봅니다.

> "`src/day3`의 ____ 파일은 ____ 계층을 담당하고, 우리 회사에서 적용하려면 ____ 부분을 가장 먼저 바꿔야 한다."

**4일차 예고** — 오늘의 State와 실행 기록을 입력으로 받아 **평가/Guardrail/Tool Plan 검증**을 다룹니다.